# Credit Card Eligibility Assessment — CrewAI + Gemini + FAISS (Hybrid RAG)

An end-to-end **multi-agent** credit-card eligibility system built on **CrewAI**, **Google Gemini**
(via the `google-genai` client and CrewAI's LiteLLM-backed `LLM`), and a **hybrid keyword + FAISS
vector search + LLM rerank** retrieval pipeline over a versioned policy corpus.

> ⚠️ You need a **Gemini API key**. In Colab, add it via **Secrets** (key icon) as `GEMINI_API_KEY`; locally,
> set the `GEMINI_API_KEY` environment variable or put it in a `.env` file.

---

## Agent design patterns covered

| Pattern | Where in this notebook |
|--------|-------------------------|
| **Prompt chaining, routing, or parallelization** — control flow beyond a single call. | Chaining: §7 `build_tasks()` (`context=[...]` handoffs). Routing: §9 `quick_prescreen()` short-circuits `AUTO_DECLINE`. Parallelization: §11 `process_batch()`, §13 `generate_letters_for_all_applicants()`. |
| **Reflection / self-critique** — the system improves its own output. | §7 `t4_critique` task (crew self-reviews its own preliminary decision). §10 `reflect_and_revise()` (draft → self-critique → revise loop for applicant letters). |
| **Tool use (≥2 tools)** or MCP integration (≥1 server). | 4 `@tool`-decorated functions, §3 `policy_retriever` (RAG); §4 `credit_bureau_lookup`, `income_verification`, `dti_calculator`. |
| **Planning (ReAct / plan-execute) or multi-agent (2+ agents with handoffs)**. | §6 `build_agents()` — 5 agents. §7 `build_tasks()` chains them via `context=[...]`; the underwriter's task (`t3_reason`) is explicitly prompted to plan Thought → Action → Observation (ReAct). |
| **Memory management (≥2 memory types)** — episodic, semantic, or a shared workspace. | §8 `build_crew()`: `memory=True` + `embedder=...` enables CrewAI's built-in **short-term** memory (Chroma, recent task context), **long-term** memory (SQLite, cross-run history), and **entity** memory (Chroma) — 3 types, ≥ the required 2. |
| **RAG** — retrieval that grounds answers in a corpus, with citations. | §1 `POLICY_CORPUS` + FAISS index. §2 `hybrid()` (keyword + vector + LLM rerank). §3 `drop_superseded()` (version resolution) + `ground()` (cites `doc_id`/`revision`/exact span, or abstains). |
| **Human-in-the-loop, guardrails, or an evaluation harness** — reliability and oversight. | §5 `eligibility_output_guardrail` (schema + fair-lending checks). §9 `route_applicant()` sets `require_human_review` for `BORDERLINE` cases, applied as `human_input=True` on the final task in §7. §12 `run_evaluation_harness()` (labeled test cases, accuracy scoring). |


## System architecture

High-level control flow for a single applicant, then the batch/eval paths.

```
                         ┌──────────────────────────┐
                         │      Applicant Input     │
                         └────────────┬─────────────┘
                                      │
                                      ▼
                         ┌──────────────────────────┐
                         │   Deterministic Router   │
                         │    quick_prescreen()     │
                         └───────┬──────┬───────────┘
                                 │      │
                    AUTO_DECLINE │      │ BORDERLINE / FAST TRACK
                                 │      │
                                 ▼      ▼
                         ┌──────────┐  ┌─────────────────────────┐
                         │ Decline  │  │      CrewAI Crew        │
                         │ No LLM   │  │                         │
                         └──────────┘  │  1. Intake Specialist   │
                                       │          ↓              │
                                       │  2. Policy Research     │
                                       │          ↓              │
                                       │  3. Senior Underwriter  │
                                       │          ↓              │
                                       │  4. Risk Reviewer       │
                                       │          ↓              │
                                       │  5. Compliance Officer  │
                                       └────────────┬────────────┘
                                                    │
                                  ┌─────────────────┼─────────────────┐
                                  │                 │                 │
                                  ▼                 ▼                 ▼
                         ┌──────────────┐  ┌──────────────┐  ┌──────────────┐
                         │ Bureau Tool  │  │ Income Tool  │  │ DTI Tool     │
                         └──────────────┘  └──────────────┘  └──────────────┘
                                  │                 │                 │
                                  └─────────────────┼─────────────────┘
                                                    ▼
                                      ┌────────────────────────┐
                                      │       Policy RAG       │
                                      │                        │
                                      │ FAISS Vector Search    │
                                      │ + Keyword Search       │
                                      │ + LLM Reranking        │
                                      │ + Revision Filtering   │
                                      └────────────┬───────────┘
                                                   │
                                                   ▼
                                      ┌────────────────────────┐
                                      │ Grounded Decision +    │
                                      │ Policy Citations       │
                                      └────────────┬───────────┘
                                                   │
                                                   ▼
                                      ┌────────────────────────┐
                                      │ Guardrail Validation   │
                                      │ + Human Review         │
                                      └────────────┬───────────┘
                                                   │
                                                   ▼
                                      ┌────────────────────────┐
                                      │ Final Decision JSON    │
                                      └────────────┬───────────┘
                                                   │
                                                   ▼
                                      ┌────────────────────────┐
                                      │ Reflection / Revision  │
                                      │ Applicant Letter       │
                                      └────────────────────────┘

## Foundation: Models, Dependencies, and Runtime Configuration

**Pattern:** System foundation

This cell installs dependencies and initializes Gemini, CrewAI, embeddings, logging,
and runtime configuration. It provides the infrastructure used by the RAG pipeline,
multi-agent workflow, tool execution, memory, and structured outputs.

In [3]:
!pip install -q crewai crewai-tools google-genai faiss-cpu numpy litellm pandas python-dotenv chromadb

##System Setup: Models, Dependencies, and Runtime Configuration

**Pattern:** System foundation for the agentic AI architecture

This cell initializes the core Python dependencies and AI services used throughout the
notebook. It configures the Gemini API, CrewAI LLMs, embedding model, structured JSON
generation, vector embeddings, logging, and runtime behavior.

### What this cell does

- Imports the libraries required for:
  - Gemini model interaction
  - CrewAI multi-agent orchestration
  - FAISS-based RAG
  - numerical and tabular processing
  - environment and secret management
- Loads the `GEMINI_API_KEY` securely from Google Colab Secrets.
- Creates the Gemini client used for generation, reranking, and embeddings.
- Configures separate Gemini models for:
  - general CrewAI reasoning;
  - lightweight/fast CrewAI tasks;
  - direct Gemini generation and reranking;
  - document and query embeddings.
- Defines `_call()` with retry handling for HTTP 429 rate limits.
- Defines `generate_json()` for schema-constrained structured outputs using Pydantic.
- Defines `embed()` to create L2-normalized vectors for semantic retrieval.
- Suppresses unnecessary CrewAI/LiteLLM tracing and logging so later agent execution
  remains readable.

### Role in the architecture

This is the **infrastructure layer** of the system. It does not perform underwriting
itself; instead, it establishes the model, embedding, structured-output, and runtime
capabilities used by the downstream **RAG, tool-use, planning, multi-agent,
guardrail, and reflection** components.

**Flow:**  
`Dependencies → Gemini/CrewAI Models → Structured Generation + Embeddings → Agentic Pipeline`

In [4]:
import os, json, re, asyncio, time
import numpy as np
import faiss
import pandas as pd
from typing import Literal
from dotenv import load_dotenv
from google import genai
from google.genai import types, errors
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from pydantic import BaseModel

load_dotenv()

# --- Gemini key --------------------------------------------------------------
load_dotenv()
api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY. Add it in Colab Secrets (key icon) and enable notebook access.")


client = genai.Client(api_key=api_key)
MODEL = "gemini-3.1-flash-lite"         # generation + reranking (raw google-genai client)
EMBED_MODEL = "gemini-embedding-001"    # turns text into vectors

# CrewAI-facing LLMs, via LiteLLM's `gemini/` provider prefix
llm = LLM(model="gemini/gemini-3.5-flash", api_key=api_key, temperature=0.3, num_retries=5)
flash_llm = LLM(model="gemini/gemini-3.1-flash-lite", api_key=api_key, temperature=0.3, num_retries=5)


# Keep output clean: quiet CrewAI/LiteLLM ERROR logs, hide a legacy-SDK notice, no run traces.
import logging, warnings
for _n in ("crewai.flow.runtime", "LiteLLM", "litellm", "root"):
    logging.getLogger(_n).setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", message=r".*google\.generativeai.*")
os.environ["CREWAI_TRACING_ENABLED"] = "false"

# Silence CrewAI's tracing notices (the repeating "Tracing Preference" panels) so only the clean
# execution boxes show. The first-time flag is captured when crewai is imported, so we also clear
# it directly on the listener. (Internal API -- guarded.)
try:
    from crewai.events.listeners.tracing.utils import (
        set_suppress_tracing_messages, mark_first_execution_done,
    )
    set_suppress_tracing_messages(True)
    mark_first_execution_done()
    from crewai.events.listeners.tracing.trace_listener import TraceCollectionListener
    if TraceCollectionListener._instance is not None:
        TraceCollectionListener._instance.first_time_handler.is_first_time = False
except Exception:
    pass


def _call(**kwargs):
    """Call Gemini, waiting once and retrying on a rate limit (HTTP 429)."""
    for attempt in range(2):
        try:
            return client.models.generate_content(model=MODEL, **kwargs)
        except errors.ClientError as e:
            if getattr(e, "code", None) == 429 and attempt == 0:
                print("  Rate limited (HTTP 429). Waiting 30s, then retrying...")
                time.sleep(30)
            else:
                raise


def generate_json(prompt, schema, system=None):
    """Structured output -- the model must return an instance of `schema`, no prose parsing."""
    cfg = types.GenerateContentConfig(system_instruction=system,
                                       response_mime_type="application/json",
                                       response_schema=schema, temperature=0)
    return _call(contents=prompt, config=cfg).parsed


def embed(texts, task):
    """Embed a list of strings and L2-normalize, so a dot product IS cosine similarity.
    task is 'SEMANTIC_SIMILARITY' (comparing words), 'RETRIEVAL_DOCUMENT' (indexing passages),
    or 'RETRIEVAL_QUERY' (a search query).
    """
    r = client.models.embed_content(model=EMBED_MODEL, contents=texts,
                                     config=types.EmbedContentConfig(task_type=task))
    v = np.array([e.values for e in r.embeddings], dtype="float32")
    v /= np.linalg.norm(v, axis=1, keepdims=True)
    return v


print("Setup ready")


Setup ready


## 1. RAG: Policy Corpus, Embeddings, and Vector Index

**Pattern:** RAG — retrieval that grounds answers in a corpus

This cell creates the authoritative credit-policy corpus and embeds each policy passage
using `gemini-embedding-001`. FAISS stores the normalized embeddings so policy questions
can be answered by retrieving relevant source passages rather than relying solely on
the language model's parametric knowledge.

The corpus intentionally contains multiple revisions of `POL-003` to demonstrate
policy-version handling later in the pipeline.


In [5]:
POLICY_CORPUS = [
    {"doc_id": "POL-001", "section": "Minimum Age & Residency","revision": "2020-01", "effective": "2020-01-01",
     "text": "Applicants must be at least 18 years old and a resident of the country of issuance. "
             "Non-residents require additional KYC documentation."},
    {"doc_id": "POL-002", "section": "Minimum Credit Score","revision": "2020-01", "effective": "2020-01-01",
     "text": "A minimum FICO-equivalent credit score of 650 is required for the Standard card. "
             "The Platinum card requires a score of 750 or above."},
    {"doc_id": "POL-003", "section": "Income Requirements","revision": "2020-01", "effective": "2020-01-01",
     "text": "Minimum verified annual income is $25,000 for the Standard card and $60,000 for the Platinum card. "
             "Self-employed applicants must provide two years of tax returns."},
    {"doc_id": "POL-003", "section": "Income Requirements","revision": "2025-01", "effective": "2025-01-01",
     "text": "Minimum verified annual income is $35,000 for the Standard card and $80,000 for the Platinum card. "
             "Self-employed applicants must provide two years of tax returns."},
    {"doc_id": "POL-004", "section": "Debt-to-Income Ratio","revision": "2020-01", "effective": "2020-01-01",
     "text": "Applicants with a debt-to-income (DTI) ratio above 45% are automatically declined regardless of "
             "credit score. DTI between 36% and 45% requires manual underwriting review."},
    {"doc_id": "POL-005", "section": "Bankruptcy & Delinquency","revision": "2020-01", "effective": "2020-01-01",
     "text": "Any active bankruptcy or delinquency within the last 24 months results in automatic decline. "
             "Discharged bankruptcies older than 4 years may be considered case-by-case."},
    {"doc_id": "POL-006", "section": "Fraud & Watchlist Screening","revision": "2020-01", "effective": "2020-01-01",
     "text": "Applicants flagged on an internal fraud watchlist or sanctions list must be declined and escalated "
             "to the compliance team, no exceptions."},
    {"doc_id": "POL-007", "section": "Fair Lending & Non-Discrimination","revision": "2020-01", "effective": "2020-01-01",
     "text": "Eligibility decisions must not use race, gender, religion, national origin, or marital status as a "
             "factor, directly or indirectly, in accordance with the Equal Credit Opportunity Act."},
    {"doc_id": "POL-008", "section": "Borderline / Manual Review Cases","revision": "2020-01", "effective": "2020-01-01",
     "text": "Applications where credit score is within 20 points of the minimum threshold, or DTI is between "
             "36-45%, must be routed to a human underwriter before final approval."},
]

TODAY = "2026-08-01"

print(f"{len(POLICY_CORPUS)} chunks. Each is one idea with its section + revision + effective date:\n")
for i, c in enumerate(POLICY_CORPUS):
    tag = "  (superseded)" if c["doc_id"] == "POL-003" and c["revision"] == "2020-01" else ""
    print(f"  [{i}] {c['section']:6} rev {c['revision']}  eff {c['effective']}{tag}")
    print(f"       {c['text']}")

# --- Policy retrieval (FAISS) -------------------------------------------------
DOC_VECS = embed([c["text"] for c in POLICY_CORPUS], "RETRIEVAL_DOCUMENT")
index = faiss.IndexFlatIP(DOC_VECS.shape[1])
index.add(DOC_VECS)
print(f"Indexed {index.ntotal} chunks, each a {DOC_VECS.shape[1]}-dimensional vector.")


9 chunks. Each is one idea with its section + revision + effective date:

  [0] Minimum Age & Residency rev 2020-01  eff 2020-01-01
       Applicants must be at least 18 years old and a resident of the country of issuance. Non-residents require additional KYC documentation.
  [1] Minimum Credit Score rev 2020-01  eff 2020-01-01
       A minimum FICO-equivalent credit score of 650 is required for the Standard card. The Platinum card requires a score of 750 or above.
  [2] Income Requirements rev 2020-01  eff 2020-01-01  (superseded)
       Minimum verified annual income is $25,000 for the Standard card and $60,000 for the Platinum card. Self-employed applicants must provide two years of tax returns.
  [3] Income Requirements rev 2025-01  eff 2025-01-01
       Minimum verified annual income is $35,000 for the Standard card and $80,000 for the Platinum card. Self-employed applicants must provide two years of tax returns.
  [4] Debt-to-Income Ratio rev 2020-01  eff 2020-01-01
       Applic

###RAG Retrieval: Semantic Vector Search
**Pattern:** RAG

This cell implements semantic retrieval using FAISS cosine similarity. A natural-language
question is converted into a query embedding and compared against the policy embeddings.

This demonstrates the basic retrieval stage of a Retrieval-Augmented Generation pipeline:
**question → embedding → nearest policy passages**.

In [6]:
def search(query, k=3):
    """Return the k nearest chunks as (score, chunk_index) pairs."""
    qv = embed([query], "RETRIEVAL_QUERY")  #Converts the input query text into a vector embedding
    scores, ids = index.search(qv, k)       #Performs a nearest-neighbor lookup in a vector index using vector qv
    return list(zip(scores[0].tolist(), ids[0].tolist())) #Unpacks the batched results (index [0]), converts NumPy array outputs into Python standard data types, and pairs them into a list of tuples


print("Query: 'what is the minimum credit score required for an applicant?'\n")
for rank, (score, i) in enumerate(search("what is the minimum credit score required for an applicant?"), 1): #Loops through the returned tuple list, assigning rank numbers starting at 1
    print(f"  {rank}. cos={score:.3f}  {POLICY_CORPUS[i]['section']}  {POLICY_CORPUS[i]['text']}") #Uses document index i to retrieve the original policy section identifier and text from the POLICY_CORPUS list.


Query: 'what is the minimum credit score required for an applicant?'

  1. cos=0.734  Minimum Credit Score  A minimum FICO-equivalent credit score of 650 is required for the Standard card. The Platinum card requires a score of 750 or above.
  2. cos=0.729  Debt-to-Income Ratio  Applicants with a debt-to-income (DTI) ratio above 45% are automatically declined regardless of credit score. DTI between 36% and 45% requires manual underwriting review.
  3. cos=0.729  Borderline / Manual Review Cases  Applications where credit score is within 20 points of the minimum threshold, or DTI is between 36-45%, must be routed to a human underwriter before final approval.


### Advanced RAG: Hybrid Retrieval and LLM Reranking

**Pattern:** RAG + prompt chaining

This cell combines two retrieval strategies:

1. Lexical keyword retrieval
2. Semantic vector retrieval

The candidate passages are merged and then passed to an LLM reranker.

The resulting pipeline is:

**Query → Keyword Search + Vector Search → Candidate Set → LLM Reranking → Best Passages**

This is particularly useful for queries containing exact identifiers such as `POL-003`,
where pure semantic retrieval may not reliably prioritize the desired passage.


In [7]:
#This code implements a simple lexical (keyword) search engine based on exact token overlap.
#Instead of using vector embeddings or semantic meaning, it counts how many unique words
#in the search query appear in each document's text and section ID.

def keyword_search(query, k=5):
    """Cheap lexical overlap -- rewards exact tokens like section numbers."""
    q = set(query.lower().replace("?", "").split())
    scored = []
    for i, c in enumerate(POLICY_CORPUS):
        toks = set((c["section"] + " " + c["text"]).lower().split())
        scored.append((len(q & toks), i))
    return [(s, i) for s, i in sorted(scored, reverse=True)[:k] if s > 0]


class RerankScore(BaseModel):
    id: int          # the candidate number shown in the prompt
    relevance: float # 0..1: how well this passage answers the question


def rerank(query, cand_ids):
    """Ask the model to score each candidate against the question; return (score, chunk_index), best first."""
    listing = "\n".join(f"[{n}] {POLICY_CORPUS[i]['section']}: {POLICY_CORPUS[i]['text']}"
                         for n, i in enumerate(cand_ids))
    scores = generate_json(
        f"""Question: {query}

Score each passage from 0.0 (irrelevant) to 1.0 (directly answers the question):
{listing}""",
        list[RerankScore],
        system="You are a senior retrieval reranker. Judge only whether each passage answers the question.",
    )
    by_n = {s.id: s.relevance for s in scores}
    return sorted(((by_n.get(n, 0.0), i) for n, i in enumerate(cand_ids)), reverse=True)


def hybrid(query, k=3, current_only=False):
    """Merge keyword + vector candidates, drop superseded if asked, then rerank."""
    cand = {i for _, i in keyword_search(query, 5)} | {i for _, i in search(query, 5)}
    if current_only:
        cand = drop_superseded(cand)
    ranked = rerank(query, sorted(cand))
    return ranked[:k], ranked


# The exact-identifier query that vector search fumbled -- hybrid + rerank recovers it.
top, _ = hybrid("what is the minimum credit score required for an applicant?", k=3)
print("Hybrid + rerank for 'what is the minimum credit score required for an applicant?':\n")
for rank, (score, i) in enumerate(top, 1):
    print(f"  {rank}. rerank={score:.2f}  {POLICY_CORPUS[i]['section']} rev {POLICY_CORPUS[i]['revision']}  {POLICY_CORPUS[i]['text']}")


Hybrid + rerank for 'what is the minimum credit score required for an applicant?':

  1. rerank=1.00  Minimum Credit Score rev 2020-01  A minimum FICO-equivalent credit score of 650 is required for the Standard card. The Platinum card requires a score of 750 or above.
  2. rerank=0.50  Borderline / Manual Review Cases rev 2020-01  Applications where credit score is within 20 points of the minimum threshold, or DTI is between 36-45%, must be routed to a human underwriter before final approval.
  3. rerank=0.20  Debt-to-Income Ratio rev 2020-01  Applicants with a debt-to-income (DTI) ratio above 45% are automatically declined regardless of credit score. DTI between 36% and 45% requires manual underwriting review.


### RAG Reliability: Version Control, Citations, and Abstention

**Pattern:** RAG + guardrails

This cell adds production-oriented safeguards to retrieval.

It:

- removes superseded policy revisions;
- respects policy effective dates;
- requires a relevance threshold;
- distinguishes `answered` from `not_covered`;
- produces structured citations;
- prevents the model from guessing when the corpus does not contain a governing rule.

The resulting flow is:

**Retrieve → Filter Current Policy → Rerank → Relevance Gate → Grounded Answer + Citation**


In [8]:
def drop_superseded(cand_ids, as_of=TODAY):
    """Per doc_id, keep only the newest revision whose effective date is on or before `as_of`."""
    best = {}
    for i in cand_ids:
        c = POLICY_CORPUS[i]
        if c["effective"] <= as_of:
            cur = best.get(c["doc_id"])
            if cur is None or c["effective"] > POLICY_CORPUS[cur]["effective"]:
                best[c["doc_id"]] = i
    return set(best.values())


class Citation(BaseModel):
    doc_id: str
    section: str
    revision: str
    span: str          # the exact text the answer relies on


class GroundedAnswer(BaseModel):
    status: Literal["answered", "not_covered"]   # answered = governed by a passage (even if the answer is "no")
    answer: str
    citation: Citation | None = None


RELEVANCE_BAR = 0.5


def ground(question, k=3, current_only=True):
    """Retrieve -> rerank -> answer with a citation, or abstain if nothing clears the bar."""
    top, ranked = hybrid(question, k=k, current_only=current_only)
    if not top or top[0][0] < RELEVANCE_BAR:
        return GroundedAnswer(status="not_covered",
                               answer="No policy corpus passage governs this question.", citation=None), ranked
    passages = "\n".join(
        f"[{POLICY_CORPUS[i]['doc_id']} | {POLICY_CORPUS[i]['section']} rev {POLICY_CORPUS[i]['revision']}] "
        f"{POLICY_CORPUS[i]['text']}"
        for _, i in top
    )
    ans = generate_json(
        f"""Passages retrieved from the credit policy corpus:
{passages}

Question: {question}

Answer ONLY from these passages. If a passage governs the question -- even if the answer is "no" --
set status "answered", give the answer, and cite the doc_id, section, revision, and the exact span you used.
If no passage governs the question, set status "not_covered" and leave citation null. Never guess.""",
        GroundedAnswer,
        system="You are a senior support analyst who answers only from cited policy corpus passages.",
    )

    return ans, ranked


def show(question, result):
    ans, _ = result
    print("Q:", question)
    if ans.status == "answered" and ans.citation:
        print(f"   ANSWERED : {ans.answer}")
        print(f"   CITES    : {ans.citation.doc_id} | {ans.citation.section} rev {ans.citation.revision} "
              f"-- \"{ans.citation.span}\"")
    else:
        print(f"   NOT COVERED : {ans.answer}")
    print()


show("What is the minimal annual income required for a Standard card?",
     ground("What is the minimal annual income required for a Standard card?"))


# --- Expose `ground()` as a CrewAI tool ---------------------------------------
# `ground()` returns a (GroundedAnswer, ranked) tuple -- not something an agent can consume directly.
# This wraps it in a proper @tool function that returns a plain string, so it can be attached to agents.
@tool("policy_retriever")
def policy_retriever(question: str) -> str:
    """Retrieves the most relevant CURRENTLY EFFECTIVE credit-card eligibility policy passages for a
    question via hybrid (keyword + vector) search with LLM reranking, and returns a grounded answer
    with its citation (doc_id, section, revision, exact span) -- or reports that no policy governs the
    question. Always use this before stating a rule or a numeric threshold."""
    ans, _ = ground(question)
    if ans.status == "answered" and ans.citation:
        return (f"ANSWER: {ans.answer}\n"
                f"CITATION: {ans.citation.doc_id} | {ans.citation.section} rev {ans.citation.revision} "
                f"-- \"{ans.citation.span}\"")
    return f"NOT COVERED: {ans.answer}"


# sanity check (no crew needed) -- @tool-wrapped functions are invoked via .run(**kwargs)
print(policy_retriever.run(question="What DTI ratio triggers automatic decline?"))


Q: What is the minimal annual income required for a Standard card?
   ANSWERED : The minimum verified annual income required for the Standard card is $35,000.
   CITES    : POL-003 | Income Requirements rev 2025-01 -- "Minimum verified annual income is $35,000 for the Standard card"

ANSWER: Applicants with a debt-to-income (DTI) ratio above 45% are automatically declined regardless of credit score.
CITATION: POL-004 | Debt-to-Income Ratio rev 2020-01 -- "Applicants with a debt-to-income (DTI) ratio above 45% are automatically declined regardless of credit score."


##2. Tool Use: Applicant Evidence and Deterministic Calculations

**Pattern:** Tool use — acting on external state

This cell provides three applicant-level tools:

- `credit_bureau_lookup`
- `income_verification`
- `dti_calculator`

The agents use these tools to obtain applicant-specific evidence.

This separates **facts obtained from tools** from **reasoning performed by the LLM**,
reducing the need for the model to invent or calculate critical underwriting data.


In [9]:
SIMULATED_BUREAU_DB={
    "APP-1007": {"credit_score": 640, "open_accounts": 2, "delinquencies_24mo": 0, "bankruptcy": False, "on_watchlist": False},
    "APP-1008": {"credit_score": 760, "open_accounts": 5, "delinquencies_24mo": 0, "bankruptcy": False, "on_watchlist": False},
    "APP-1009": {"credit_score": 710, "open_accounts": 4, "delinquencies_24mo": 0, "bankruptcy": True,  "on_watchlist": False},
    "APP-1010": {"credit_score": 660, "open_accounts": 3, "delinquencies_24mo": 0, "bankruptcy": False, "on_watchlist": False},
    "APP-1011": {"credit_score": 735, "open_accounts": 1, "delinquencies_24mo": 0, "bankruptcy": False, "on_watchlist": True},
}


SIMULATED_INCOME_DB={
    "APP-1007": {"annual_income": 45000, "monthly_debt": 1800, "employment": "salaried"},       # DTI = 48.0% (Exceeds 45%)
    "APP-1008": {"annual_income": 90000, "monthly_debt": 3000, "employment": "salaried"},       # DTI = 40.0% (Borderline DTI 36-45%)
    "APP-1009": {"annual_income": 85000, "monthly_debt": 1000, "employment": "salaried"},       # High score & safe DTI, but Bankruptcy flag
    "APP-1010": {"annual_income": 50000, "monthly_debt": 1000, "employment": "salaried"},       # Score = 660 (Within 20 pts of 650 threshold)
    "APP-1011": {"annual_income": 120000, "monthly_debt": 1500, "employment": "salaried"},      # High score & low DTI, but Watchlist flag
}

@tool("credit_bureau_lookup")
def credit_bureau_lookup(applicant_id: str) -> str:
    """Looks up an applicant's credit bureau record: credit score, delinquencies, bankruptcy flag,
    fraud watchlist status. Input: applicant_id, e.g. 'APP-1001'."""
    record = SIMULATED_BUREAU_DB.get(applicant_id)
    return json.dumps(record) if record else f"No bureau record found for {applicant_id}."

@tool("income_verification")
def income_verification(applicant_id: str) -> str:
    """Retrieves verified annual income, monthly debt obligations, and employment type for an
    applicant. Input: applicant_id, e.g. 'APP-1001'."""
    record = SIMULATED_INCOME_DB.get(applicant_id)
    return json.dumps(record) if record else f"No income record found for {applicant_id}."

@tool("dti_calculator")
def dti_calculator(annual_income: float, monthly_debt: float) -> str:
    """Calculates debt-to-income (DTI) ratio given annual income and monthly debt payments."""
    monthly_income = annual_income / 12
    if monthly_income <= 0:
        return "Invalid income."
    return f"DTI ratio = {(monthly_debt / monthly_income) * 100:.1f}%"

# sanity checks (no LLM needed) -- @tool-wrapped functions are invoked via .run(**kwargs)
print("=============Sanity Checks==================")
print(credit_bureau_lookup.run(applicant_id="APP-1007"))
print(income_verification.run(applicant_id="APP-1009"))
print(dti_calculator.run(annual_income=31000, monthly_debt=1500))


=============Sanity Checks==================
{"credit_score": 640, "open_accounts": 2, "delinquencies_24mo": 0, "bankruptcy": false, "on_watchlist": false}
{"annual_income": 85000, "monthly_debt": 1000, "employment": "salaried"}
DTI ratio = 58.1%


##3. Guardrails: Structured Decisions and Fair-Lending Controls

**Pattern:** Guardrails / reliability and oversight

This cell validates the final underwriting output before CrewAI accepts it.

The guardrail checks:

- valid JSON;
- required fields;
- permitted decision values;
- presence of policy citations;
- prohibited protected-attribute language.

The purpose is to place a deterministic control boundary around the final LLM-generated
decision.

In [10]:
REQUIRED_KEYS = {"applicant_id", "decision", "card_tier", "reasons", "citations", "confidence"}
PROHIBITED_TERMS = ["race", "gender", "religion", "ethnicity", "national origin", "marital status"]

def eligibility_output_guardrail(task_output):
    """Validates the final decision JSON schema and screens for fair-lending violations
    before the task result is accepted. Returns (success: bool, result_or_feedback)."""
    raw = task_output.raw if hasattr(task_output, "raw") else str(task_output)
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return (False, f"Output is not valid JSON. Return ONLY a JSON object with keys: {sorted(REQUIRED_KEYS)}.")
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError as e:
        return (False, f"JSON parse error: {e}. Fix formatting and return valid JSON only.")

    missing = REQUIRED_KEYS - set(data.keys())
    if missing:
        return (False, f"Missing required keys: {missing}. Include all of {sorted(REQUIRED_KEYS)}.")

    if data["decision"] not in {"APPROVE", "DECLINE", "MANUAL_REVIEW"}:
        return (False, "decision must be one of APPROVE, DECLINE, MANUAL_REVIEW.")

    reasons_text = " ".join(data.get("reasons", [])).lower()
    for term in PROHIBITED_TERMS:
        if term in reasons_text:
            return (False, f"Fair-lending violation: reasoning references protected attribute '{term}'. "
                            "Remove it and base the decision only on credit/income/policy factors.")

    if not data.get("citations"):
        return (False, "Decision must include at least one policy citation (doc_id).")

    return (True, data)


##4. Multi-Agent Architecture

**Pattern:** Multi-agent collaboration — specialized agents with handoffs

This cell defines the specialized agents in the system:

| Agent | Responsibility |
|---|---|
| Application Intake Specialist | Extract applicant information |
| Policy Research Analyst | Retrieve applicable policy |
| Senior Underwriter | Plan investigation and evaluate evidence |
| Independent Risk Reviewer | Critique the preliminary decision |
| Compliance Officer | Produce the final guarded decision |

Each agent has a specialized role and receives only the tools and responsibilities
needed for that role.


In [11]:
def build_agents():
    """Returns a FRESH set of agent instances. Required for safe concurrent execution: building
    new Agent objects per crew run (instead of module-level singletons reused across every
    build_crew() call) ensures no two crews running at the same time ever share an agent's
    internal executor."""
    extraction_agent = Agent(
        role="Application Intake Specialist",
        goal="Parse raw applicant submissions into clean, structured data.",
        backstory="A meticulous operations analyst who has processed thousands of card applications and never lets a missing field slip through.",
        llm=flash_llm,
        verbose=True,
        allow_delegation=False,
    )
    policy_agent = Agent(
        role="Policy Research Analyst",
        goal="Ground every eligibility question in the official credit policy corpus and always cite doc_ids.",
        backstory="A compliance researcher who refuses to state a rule without pointing to the exact policy clause.",
        llm=llm,
        tools=[policy_retriever],
        verbose=True,
        allow_delegation=False,
    )
    reasoning_agent = Agent(
        role="Senior Underwriter (ReAct planner)",
        goal=("Reason step by step (Thought -> Action -> Observation) using bureau, income, DTI and policy tools to "
              "reach a defensible eligibility decision."),
        backstory=("A senior underwriter who plans each investigative step before acting, gathers evidence with tools, "
                   "and never guesses a number that a tool can verify."),
        llm=llm,
        tools=[credit_bureau_lookup, income_verification, dti_calculator, policy_retriever],
        verbose=True,
        allow_delegation=False,
    )
    critic_agent = Agent(
        role="Independent Risk Reviewer",
        goal=("Critique the underwriter's decision for policy compliance, math errors, missing citations, and "
              "fair-lending risk. Suggest concrete fixes."),
        backstory="A skeptical second-line-of-defense reviewer who has caught dozens of underwriting mistakes before they reached customers.",
        llm=llm,
        tools=[policy_retriever],
        verbose=True,
        allow_delegation=False,
    )
    compliance_officer = Agent(
        role="Compliance Officer",
        goal="Issue the final, guardrail-checked eligibility decision in strict JSON, escalating borderline cases to a human.",
        backstory="Final sign-off authority who is personally accountable for every card issued or declined.",
        llm=llm,
        verbose=True,
        allow_delegation=False,
    )
    return extraction_agent, policy_agent, reasoning_agent, critic_agent, compliance_officer


##5. Prompt Chaining: Sequential Agent Workflow
**Pattern:** Prompt chaining + multi-agent handoffs

This cell connects the agents into a sequential workflow:

**Extraction → Policy Research → Underwriting → Critique → Final Decision**

Each task consumes the output of previous tasks through `context=[...]`.

This creates a multi-step reasoning pipeline rather than a single prompt/response call.

In [12]:


def build_tasks(applicant_raw_text: str, agents: tuple, require_human_review: bool = False):
    extraction_agent, policy_agent, reasoning_agent, critic_agent, compliance_officer = agents

    t1_extract = Task(
        description=(f"Extract structured applicant fields (applicant_id, requested_card_tier, self-reported "
                      f"details) from this raw submission:\n\n{applicant_raw_text}\n\n"
                      "Return a short structured summary."),
        expected_output="A structured summary listing applicant_id, requested_card_tier, and any self-reported facts.",
        agent=extraction_agent,
    )

    t2_policy = Task(
        description=("Using the applicant summary from the previous step, retrieve and summarize ALL policy "
                      "clauses relevant to this applicant's requested card tier: age/residency, minimum credit "
                      "score, income requirements, DTI thresholds, bankruptcy rules, and fraud screening. "
                      "Cite every clause by doc_id."),
        expected_output="A bullet list of relevant policy clauses, each with a doc_id citation.",
        agent=policy_agent,
        context=[t1_extract],
    )

    t3_reason = Task(
        description=("Act as a ReAct-style planner: for each policy requirement identified in the previous step, "
                      "decide which tool to call (credit_bureau_lookup, income_verification, dti_calculator) to "
                      "verify it, call the tool, observe the result, and record whether the applicant PASSES or "
                      "FAILS that requirement. After checking all requirements, propose a preliminary decision "
                      "(APPROVE / DECLINE / MANUAL_REVIEW) with supporting evidence and policy citations."),
        expected_output="A step-by-step Thought/Action/Observation trace ending in a preliminary decision with evidence and citations.",
        agent=reasoning_agent,
        context=[t1_extract, t2_policy],
    )

    t4_critique = Task(
        description=("Critically review the underwriter's preliminary decision and evidence trace. Check: "
                      "(a) every numeric claim was tool-verified, (b) every policy claim has a doc_id citation, "
                      "(c) no protected attributes (race, gender, religion, national origin, marital status) "
                      "influenced the reasoning, (d) borderline thresholds (DTI 36-45%, score within 20 pts of "
                      "cutoff) are flagged for MANUAL_REVIEW per POL-008. List any problems found, or state "
                      "'NO ISSUES FOUND' if the decision is sound."),
        expected_output="Either 'NO ISSUES FOUND' or a list of specific problems with the preliminary decision.",
        agent=critic_agent,
        context=[t1_extract, t2_policy, t3_reason],
    )

    t5_decide = Task(
        description=("Using the underwriter's reasoning and the reviewer's critique, issue the FINAL decision. "
                      "If the critique found issues, resolve them. Respond with ONLY a JSON object with keys: "
                      "applicant_id, decision (APPROVE/DECLINE/MANUAL_REVIEW), card_tier, reasons (list of strings, "
                      "no protected-attribute language), citations (list of doc_ids), confidence (0-1 float)."),
        expected_output="A single JSON object with keys applicant_id, decision, card_tier, reasons, citations, confidence.",
        agent=compliance_officer,
        context=[t1_extract, t2_policy, t3_reason, t4_critique],
        guardrail=eligibility_output_guardrail,
        human_input=require_human_review,  # HUMAN-IN-THE-LOOP for borderline cases (Pattern 7)
    )

    return [t1_extract, t2_policy, t3_reason, t4_critique, t5_decide]


##6. Memory Management and Crew Orchestration

**Pattern:** Memory management + multi-agent orchestration

This cell constructs a fresh CrewAI crew for every applicant.

CrewAI memory is enabled and configured with a Gemini embedding model. The architecture
therefore provides semantic memory infrastructure alongside the explicit policy RAG
system.

Fresh agent instances are intentionally created for each crew so concurrent executions
do not share executor state.

In [13]:
def build_crew(applicant_raw_text: str, require_human_review: bool = False, verbose: bool = True) -> Crew:
    # Fresh agents every call -- see the CONCURRENCY NOTE in §6. Sharing global singleton agents
    # across crews that run at the same time raises "Executor is already running. Cannot invoke
    # the same executor instance concurrently."
    agents = build_agents()
    tasks = build_tasks(applicant_raw_text, agents, require_human_review=require_human_review)

    # Modern CrewAI embedder configuration format for Gemini; CrewAI manages the underlying
    # short-term/entity (Chroma) and long-term (SQLite) memory storage automatically.
    embedder_config = {
        "provider": "google-generativeai",
        "config": {
            "api_key": api_key,
            "model": "gemini-embedding-001"
        }
    }

    try:
        return Crew(
            agents=list(agents),
            tasks=tasks,
            process=Process.sequential,
            memory=True,
            embedder=embedder_config,
            verbose=verbose,
        )
    except Exception as e:
        print(f"[memory] Custom embedder setup failed ({e}); falling back to default memory.")
        return Crew(agents=list(agents), tasks=tasks, process=Process.sequential, memory=True, verbose=verbose)


##7. Routing: Deterministic Agentic Control Flow

**Pattern:** Routing — conditional execution

This cell implements a low-cost deterministic router before invoking the LLM workflow.

Applicants are routed into:

- `AUTO_DECLINE`
- `BORDERLINE`
- `FAST_TRACK_APPROVE`

Hard-decline conditions avoid unnecessary LLM calls, while borderline cases can be
escalated to additional review.

This demonstrates that agentic systems do not need to invoke an LLM for every request.


In [14]:
def quick_prescreen(applicant_id: str) -> dict:
    """Non-LLM heuristic ROUTER used before invoking the expensive multi-agent crew."""
    bureau = SIMULATED_BUREAU_DB.get(applicant_id, {})
    income = SIMULATED_INCOME_DB.get(applicant_id, {})
    score = bureau.get("credit_score", 0)
    monthly_income = (income.get("annual_income", 0) / 12) if income else 0
    dti = (income.get("monthly_debt", 0) / monthly_income * 100) if monthly_income else 100

    #Routing
    if bureau.get("on_watchlist") or bureau.get("bankruptcy") or bureau.get("delinquencies_24mo", 0) > 0:
        tier = "AUTO_DECLINE"
    elif dti > 45:
        tier = "AUTO_DECLINE"
    elif (36 <= dti <= 45) or (abs(score - 650) <= 20) or (abs(score - 750) <= 20):
        tier = "BORDERLINE"
    elif score >= 650:
        tier = "FAST_TRACK_APPROVE"
    else:
        tier = "AUTO_DECLINE"
    return {"tier": tier, "credit_score": score, "dti": round(dti, 1)}


async def route_applicant(applicant_id: str, applicant_raw_text: str, verbose: bool = True) -> dict:
    """ROUTING: decides how much agent work an applicant needs, and whether a human must sign off.

    NOTE: this is async and uses `crew.kickoff_async()` rather than `crew.kickoff()`. Notebook
    kernels (Jupyter/Colab) already run their own asyncio event loop, and recent CrewAI versions
    explicitly refuse to run the synchronous `kickoff()` from inside a running loop -- you'll hit
    "Agent execution was invoked synchronously from within a running event loop" otherwise. Call
    this with `await route_applicant(...)` (notebook cells support top-level `await`).
    """
    prescreen = quick_prescreen(applicant_id)
    print(f"[router] {applicant_id} -> {prescreen}")

    if prescreen["tier"] == "AUTO_DECLINE":
        return {
            "applicant_id": applicant_id, "decision": "DECLINE", "card_tier": "N/A",
            "reasons": ["Failed automatic hard-decline screen (watchlist, bankruptcy, delinquency, or DTI > 45%)."],
            "citations": ["POL-004", "POL-005", "POL-006"],
            "confidence": 0.99, "route": "AUTO_DECLINE (no LLM call)",
        }

    require_human = prescreen["tier"] == "BORDERLINE"
    crew = build_crew(applicant_raw_text, require_human_review=require_human, verbose=verbose)
    result = await crew.kickoff_async()
    return {"applicant_id": applicant_id, "route": prescreen["tier"], "raw_result": result}


# Sanity check the router alone (no LLM needed)
for aid in SIMULATED_BUREAU_DB:
    print(aid, "->", quick_prescreen(aid))


APP-1007 -> {'tier': 'AUTO_DECLINE', 'credit_score': 640, 'dti': 48.0}
APP-1008 -> {'tier': 'BORDERLINE', 'credit_score': 760, 'dti': 40.0}
APP-1009 -> {'tier': 'AUTO_DECLINE', 'credit_score': 710, 'dti': 14.1}
APP-1010 -> {'tier': 'BORDERLINE', 'credit_score': 660, 'dti': 24.0}
APP-1011 -> {'tier': 'AUTO_DECLINE', 'credit_score': 735, 'dti': 15.0}


##8. Reflection / Self-Critique

**Pattern:** Reflection — the system improves its own output

This cell adds a second-pass quality loop to applicant correspondence.

The workflow is:

**Generate Letter → Critique Letter → Revise → Critique → Approve or Stop**

The same correspondence agent evaluates its previous output against requirements for:

- specific decline reasons;
- protected-attribute language;
- clarity;
- legal/compliance tone.

The loop is bounded by `max_iterations` to prevent uncontrolled execution.

In [15]:
async def reflect_and_revise(decision_json: dict, max_iterations: int = 2) -> str:
    """Draft an adverse-action / approval letter, then have the SAME agent critique its own
    draft and revise until it self-approves or max_iterations is hit.

    Async + `kickoff_async()` for the same reason as `route_applicant()` -- avoids CrewAI's
    "invoked synchronously from within a running event loop" error inside notebook kernels.
    Call this with `await reflect_and_revise(...)`.
    """
    letter_agent = Agent(
        role="Customer Correspondence Writer",
        goal="Write clear, compliant applicant-facing letters explaining the eligibility decision.",
        backstory="A writer who must follow Regulation B adverse-action notice requirements.",
        llm=flash_llm,
        verbose=False,
    )

    draft_task = Task(
        description=(f"Write a short applicant-facing letter for this decision: {json.dumps(decision_json, default=str)}. "
                      "If DECLINE, include the specific principal reasons (Regulation B requires this) drawn "
                      "from decision_json['reasons']. Keep it under 150 words."),
        expected_output="A short letter to the applicant.",
        agent=letter_agent,
    )
    draft_crew = Crew(agents=[letter_agent], tasks=[draft_task], process=Process.sequential, verbose=False)
    letter = str(await draft_crew.kickoff_async())

    for i in range(max_iterations):
        critique_task = Task(
            description=(f"Self-critique this letter for: (1) missing specific decline reasons, "
                         f"(2) any protected-attribute language, (3) unclear/legalese tone. "
                         f"Letter:\n\n{letter}\n\nReply 'APPROVED' if it has no issues, otherwise reply with a "
                         f"revised version of the letter only."),
            expected_output="Either 'APPROVED' or a revised letter.",
            agent=letter_agent,
        )
        critique_crew = Crew(agents=[letter_agent], tasks=[critique_task], process=Process.sequential, verbose=False)
        result = str(await critique_crew.kickoff_async()).strip()
        print(f"[reflection iter {i+1}] {'no changes needed' if result.upper().startswith('APPROVED') else 'revised draft'}")
        if result.upper().startswith("APPROVED"):
            break
        letter = result
    return letter


##9. Evaluation Harness: Reliability and Regression Testing

**Pattern:** Evaluation harness — reliability and oversight

This cell runs labeled test cases against the deterministic router and measures:

- routing accuracy;
- credit score;
- calculated DTI;
- expected route;
- automatic-decline rate.

The harness can optionally execute the full LLM workflow, turning the notebook from
a demonstration into a basic regression-testing environment.

In [16]:

TEST_CASES = [
    {
        "applicant_id": "APP-1007",
        "expected_route": "AUTO_DECLINE",
        "text": "Applicant APP-1007 requests a Standard card.",
    },
    {
        "applicant_id": "APP-1008",
        "expected_route": "BORDERLINE",
        "text": "Applicant APP-1008 requests a Platinum card.",
    },
    {
        "applicant_id": "APP-1009",
        "expected_route": "AUTO_DECLINE",
        "text": "Applicant APP-1009 requests a Platinum card."
    },
    {
        "applicant_id": "APP-1010",
        "expected_route": "BORDERLINE",
        "text": "Applicant APP-1010 requests a Standard card."
    },
    {
        "applicant_id": "APP-1011",
        "expected_route": "AUTO_DECLINE",
        "text": "Applicant APP-1011 requests a Platinum card."
    },
]


async def run_evaluation_harness(call_llm: bool = False) -> pd.DataFrame:
    """RELIABILITY / EVALUATION HARNESS: runs the router (and optionally the full LLM crew)
    against labeled test cases and reports routing accuracy -- a lightweight regression test.

    Async because it awaits `route_applicant()` when `call_llm=True`. Call with
    `await run_evaluation_harness(...)` even when `call_llm=False` (still a coroutine)."""
    rows = []
    for case in TEST_CASES:
        prescreen = quick_prescreen(case["applicant_id"])
        row = {
            "applicant_id": case["applicant_id"],
            "router_tier": prescreen["tier"],
            "credit_score": prescreen["credit_score"],
            "dti": prescreen["dti"],
            "expected_route": case["expected_route"],
            "route_correct": prescreen["tier"] == case["expected_route"],
        }
        if call_llm and prescreen["tier"] != "AUTO_DECLINE":
            out = await route_applicant(case["applicant_id"], case["text"])
            row["llm_result"] = str(out.get("raw_result", out))[:200]
        rows.append(row)

    df = pd.DataFrame(rows)
    accuracy = df["route_correct"].mean()
    auto_decline_rate = (df["router_tier"] == "AUTO_DECLINE").mean()
    print(f"Router accuracy: {accuracy:.0%}  |  Auto-decline (zero-LLM-call) rate: {auto_decline_rate:.0%}")
    return df

eval_df = await run_evaluation_harness(call_llm=False)
eval_df


Router accuracy: 100%  |  Auto-decline (zero-LLM-call) rate: 60%


,applicant_id,router_tier,credit_score,dti,expected_route,route_correct
0,APP-1007,AUTO_DECLINE,640,48.0,AUTO_DECLINE,True
1,APP-1008,BORDERLINE,760,40.0,BORDERLINE,True
2,APP-1009,AUTO_DECLINE,710,14.1,AUTO_DECLINE,True
3,APP-1010,BORDERLINE,660,24.0,BORDERLINE,True
4,APP-1011,AUTO_DECLINE,735,15.0,AUTO_DECLINE,True


## 10. End-to-End Agentic Workflow

**Patterns:** Routing + RAG + tool use + multi-agent planning + guardrails + reflection + parallelization

This cell combines the major components into an end-to-end workflow:

**Applicant → Router → Multi-Agent Crew → Decision → Reflection → Applicant Letter**

Multiple applicants can execute concurrently, while individual decisions are grounded
in policy retrieval, applicant tools, structured output validation, and compliance review.


In [17]:
async def generate_letter_for_applicant(case: dict, verbose: bool = False) -> dict:
    """Runs route_applicant() then reflect_and_revise() for ONE applicant (drawn from TEST_CASES /
    SIMULATED_BUREAU_DB / SIMULATED_INCOME_DB), returning applicant_id, route, decision, and the
    generated letter (or an error)."""

    try:
        outcome = await route_applicant(case["applicant_id"], case["text"], verbose=verbose)

        raw_result = outcome.get("raw_result")
        if raw_result is None:
            # AUTO_DECLINE short-circuit path (see route_applicant): outcome is already a plain,
            # JSON-serializable decision dict with no "raw_result" key.
            decision_for_letter = outcome
        else:
            # crew.kickoff_async() returns a CrewOutput object -- pull the text out via `.raw`
            # (falling back to str()) before parsing the JSON decision out of it.
            raw_text = getattr(raw_result, "raw", None) or str(raw_result)
            m = re.search(r"\{.*\}", raw_text, re.DOTALL)
            decision_for_letter = json.loads(m.group(0)) if m else {
                "applicant_id": case["applicant_id"], "decision": "MANUAL_REVIEW", "card_tier": "N/A",
                "reasons": ["Could not parse a structured decision from the crew's output; see raw_result."],
                "citations": [], "confidence": 0.0,
            }

        letter = await reflect_and_revise(decision_for_letter)
        return {
            "applicant_id": case["applicant_id"], "route": outcome.get("route"),
            "decision": decision_for_letter.get("decision"), "letter": letter, "error": None,
        }
    except Exception as e:
        return {
            "applicant_id": case["applicant_id"], "route": None,
            "decision": None, "letter": None, "error": str(e),
        }


async def generate_letters_for_all_applicants(cases: list = TEST_CASES) -> list:
    """Runs the full pipeline (router -> crew -> reflection letter) for every applicant in `cases`
    CONCURRENTLY via asyncio.gather -- `cases` defaults to TEST_CASES, which is built from the
    applicant_ids in SIMULATED_BUREAU_DB / SIMULATED_INCOME_DB, instead of a single hardcoded
    sample_applicant_text."""

    return await asyncio.gather(*(generate_letter_for_applicant(c) for c in cases))


try:
    results = await generate_letters_for_all_applicants(TEST_CASES)

    for r in results:
        print("=" * 70)
        print(f"{r['applicant_id']}  route={r['route']}  decision={r['decision']}")
        if r["error"]:
            print(f"  ERROR: {r['error']}")
        else:
            print("\n--- Letter ---\n")
            print(r["letter"])
        print()

    letters_df = pd.DataFrame([{
        "applicant_id": r["applicant_id"], "route": r["route"], "decision": r["decision"],
        "error": r["error"], "letter_preview": (r["letter"][:120] + "...") if r["letter"] else None,
    } for r in results])
    letters_df
except Exception as e:
    print("Demo requires a valid GEMINI_API_KEY and installed dependencies (crewai, crewai-tools, litellm). Error:")
    print(e)


[router] APP-1007 -> {'tier': 'AUTO_DECLINE', 'credit_score': 640, 'dti': 48.0}
[router] APP-1008 -> {'tier': 'BORDERLINE', 'credit_score': 760, 'dti': 40.0}


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


[router] APP-1009 -> {'tier': 'AUTO_DECLINE', 'credit_score': 710, 'dti': 14.1}
[router] APP-1010 -> {'tier': 'BORDERLINE', 'credit_score': 660, 'dti': 24.0}
[router] APP-1011 -> {'tier': 'AUTO_DECLINE', 'credit_score': 735, 'dti': 15.0}
[reflection iter 1] revised draft
[reflection iter 1] revised draft
[reflection iter 2] revised draft
[reflection iter 1] revised draft
[reflection iter 2] revised draft
[reflection iter 2] revised draft


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Application Intake Specialist                                                                           │
│                                                                                                                 │
│  Task: Extract structured applicant fields (applicant_id, requested_card_tier, self-reported details) from      │
│  this raw submission:                                                                                           │
│                                                                                                                 │
│  Applicant APP-1008 requests a Platinum card.                                                                   │
│                                                                                                                 │
│  Return a short structured summary.                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Application Intake Specialist                                                                           │
│                                                                                                                 │
│  Task: Extract structured applicant fields (applicant_id, requested_card_tier, self-reported details) from      │
│  this raw submission:                                                                                           │
│                                                                                                                 │
│  Applicant APP-1010 requests a Standard card.                                                                   │
│                                                                                                                 │
│  Return a short structured summary.                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Compliance Officer                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "applicant_id": "APP-1010",                                                                                  │
│    "decision": "MANUAL_REVIEW",                                                                                 │
│    "card_tier": "Standard",                                                                                     │
│    "reasons": [                                                                                                 │
│      "The applicant's credit score of 660 is within 20 points of the minimum threshold of 650 for the Standard  │
│  card, which triggers a mandatory manual underwriting review under borderline case guidelines.",                │
│      "Age and residency requirements are unverified via standard automated tools and require manual KYC         │
│  verification."                                                                                                 │
│    ],                                                                                                           │
│    "citations": [                                                                                               │
│      "POL-001",                                                                                                 │
│      "POL-002",                                                                                                 │
│      "POL-008"                                                                                                  │
│    ],                                                                                                           │
│    "confidence": 1.0                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── 💬 Human Feedback Required ───────────────────────────────────────────╮
│                                                                                                                 │
│  Provide feedback on the Final Result above.                                                                    │
│                                                                                                                 │
│  • If you are happy with the result, simply hit Enter without typing anything.                                  │
│  • Otherwise, provide specific improvement requests.                                                            │
│  • You can provide multiple rounds of feedback until satisfied.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Ending event 'memory_save_failed' emitted with empty scope stack. Missing starting 
event?

[reflection iter 1] revised draft
[reflection iter 2] revised draft
APP-1007  route=AUTO_DECLINE (no LLM call)  decision=DECLINE

--- Letter ---

Subject: Important information regarding your credit application

Dear Applicant,

Thank you for giving us the opportunity to review your application for a credit account (Reference: APP-1007).

We regret to inform you that we are unable to approve your application at this time. Our decision was based on the following principal reason(s):

*   [Insert specific reason, e.g., Delinquent past or present credit obligations with others]
*   [Insert specific reason, e.g., Debt-to-income ratio is too high]

In evaluating your application, we used a credit report provided by [Insert Name of Credit Reporting Agency]. The reporting agency played no part in our decision and is unable to provide you with the specific reasons why your application was denied. You have a right under the Fair Credit Reporting Act to obtain a free copy of your report from the